In [1]:
import numpy as np
import pandas as pd
import awkward as ak
import uproot
import matplotlib.pyplot as plt
from IPython.display import Image, display

import multiprocessing as mp
import os
import time

import gc
import itertools

```python
"""
Paths:
Chunk0950                    /Chunk0950/001_007/AO2Dtree.root
Chunk1000                    [ /Chunk1000/001_005/AO2Dtree.root, /Chunk1000/006_009/AO2Dtree.root ]
Chunk1010                    [ /Chunk1010/001_005/AO2Dtree.root, /Chunk1010/006_010/AO2Dtree.root ]
Chunk1020                    /Chunk1020/001_007/AO2Dtree.root
Chunk1030                    [ /Chunk1030/001_005/AO2Dtree.root, /Chunk1030/006_011/AO2Dtree.root ]
Chunk1040                    /Chunk1040/001_007/AO2Dtree.root
Chunk1050                    [ /Chunk1050/001_005/AO2Dtree.root, /Chunk1050/006_010/AO2Dtree.root ]
Chunk1100                    [ /Chunk1100/001_005/AO2Dtree.root, /Chunk1100/006_009/AO2Dtree.root ]
Chunk1110                    /Chunk1110/001_005/AO2Dtree.root
Chunk1120                    [ /Chunk1120/001_005/AO2Dtree.root, /Chunk1120/006_011/AO2Dtree.root ]
Chunk1130                    /Chunk1130/001_008/AO2Dtree.root
Chunk1140                    [ /Chunk1140/001_006/AO2Dtree.root, /Chunk1120/007_011/AO2Dtree.root ]
Chunk1150                    /Chunk1150/001_007/AO2Dtree.root
Chunk1200                    [ /Chunk1200/001_006/AO2Dtree.root, /Chunk1120/007_011/AO2Dtree.root ]
Chunk1210                    /Chunk1210/001_008/AO2Dtree.root
Chunk1220                    /Chunk1220/001_008/AO2Dtree.root
Chunk1300                    [ /Chunk1300/001_006/AO2Dtree.root, /Chunk1300/003/AO2Dtree.root, /Chunk1300/007_010/AO2Dtree.root ]
"""
```

In [2]:
# Converting from track coordinates to global (lab) coordinates.
def to_global_coord( row ):
    xy_in = np.array([row["fX"],row["fY"]])
    # angle = -row["fAlpha"]
    # rot = np.array([[np.cos(angle),np.sin(angle)], [-np.sin(angle), np.cos(angle)]]) # inverse matrix of the one from to_track_coord (-alfa)
    # Rewriting the above matrix using cosine/sin even/odd function properties: less multiplications
    angle = row["fAlpha"]
    rot = np.array([[np.cos(angle),-np.sin(angle)], [np.sin(angle), np.cos(angle)]])
    xy_out = rot.dot(xy_in)
    return( xy_out )

# Secondary vertex estimation via approximation of linear tracks.
def secondary_vertex ( row1, row2 ):
    XY1, XY2 = [to_global_coord(row1), to_global_coord(row2)]
    x1 = XY1[0]
    y1 = XY1[1]
    x2 = XY2[0]
    y2 = XY2[1]
    
    px1 = row1["px"]
    px2 = row2["px"]
    py1 = row1["py"]
    py2 = row2["py"]
    pz1 = row1["pz"]
    pz2 = row2["pz"]
    
    m1 = py1/px1
    m2 = py2/px2
    q1 = y1 - m1*x1
    q2 = y2 - m2*x2
    x_SV = ( q2 - q1 )/( m1 - m2)
    y_SV = y1 + m1*(x_SV - x1)
   
    z1_track = pz1/px1 * (x_SV - x1) + row1["fZ"]
    z2_track = pz2/px2 * (x_SV - x2) + row2["fZ"]
    z_SV = (z1_track + z2_track)/2
    return ([x_SV, y_SV, z_SV])

In [3]:
def compute_pairs(file_path, ID_split):

    # Load the file.
    with uproot.open(file_path) as file:
        # Load TTrees from directories
        names_track_extr = file.keys(filter_name=r"*O2filtertrackextr")[ID_split[0]: ID_split[-1]+1]
        names_track      = file.keys(filter_name=r"*O2filtertrack")[ID_split[0]: ID_split[-1]+1]
        names_coll       = file.keys(filter_name=r"*O2collision_001")[ID_split[0]: ID_split[-1]+1]

    # Variable to correct the index of the collision (because it starts from 0 at each new TTree).
    collision_offset = 0          
    
    # Particle masses in GeV.
    m_K = 0.493677
    m_pi = 0.139570
    m_d0 = 1.86484

    # Invariant Mass interval to keep.
    LOWER_MASS = 0.8*m_d0
    UPPER_MASS = 1.2*m_d0

    # Container for the dataframes created with the pairs.
    list_of_df = []               
    
    for i in range(len(names_coll)):
    
        # Read collision tree.
        df_coll = file[ names_coll[i] ].arrays(["fPosX", "fPosY", "fPosZ"], library="pd")   
    
        # Read track and trackextr using boolean mask for hypothesis on particle identity.
        df_trackextr = file[ names_track_extr[i] ].arrays(["fPt", "fEta", "fCharge", "fDcaXY", "fNsigmaTPCpi", "fNsigmaTPCka",
                                                           "fNsigmaTOFpi", "fNsigmaTOFka"], library="pd" )
        df_track = file[ names_track[i] ].arrays(["fIndexCollisions", "fAlpha", "fX", "fY", "fZ"], library="pd")
        
        # Merge all rows: fIndexCollisions in O2filtertrack must match the index in O2collision_001;
        # O2filtertrack and O2filtertrackextr, instead, match 1:1 row-wise.
        df_track = pd.merge(left=df_track, right=df_coll, how="inner", left_on = "fIndexCollisions", right_index=True)
        df_trackextr = pd.merge(left=df_trackextr, right=df_track, how='inner', left_index=True, right_index=True)
        
        # Assign particle nature hypothesis based on the chosen cuts.
        mask_both = df_trackextr.eval(f"({Ka_hyp} & {Pi_hyp})")
        mask_Ka = df_trackextr.eval(Ka_hyp)
        mask_Pi = df_trackextr.eval(Pi_hyp)
        
        # From the documentation of numpy select: numpy.select(condlist, choicelist, default=0)
        # When multiple conditions are satisfied, the first one encountered in condlist is used.
        df_trackextr["Hyp"] = np.select([mask_both, mask_Ka, mask_Pi], ["Both", "Kaon", "Pion"], default="Bkg")
        
        # Apply the SAME filter to df_trackextr and df_track to keep them aligned (and keep only useful columns).
        mask = mask_Ka | mask_Pi
        df_trackextr = df_trackextr.loc[mask, ["fIndexCollisions","fAlpha", "fX", "fY", "fZ","fPt", "fEta", "fCharge", "fDcaXY","Hyp",
                                               "fPosX", "fPosY", "fPosZ", "fNsigmaTPCpi", "fNsigmaTPCka", "fNsigmaTOFpi", "fNsigmaTOFka"] ]
    
        # The rows where the fIndexCollision is negative have been excluded thanks to the merge on the index of collision, which can only be
        # non-negative. So, we keep only those with |fPosZ| < 10 and |pseudorapidity| < 0.8
        valid = ( df_trackextr["fPosZ"].abs() < 10 ) & ( df_trackextr["fEta"].abs() < 0.8 )
        df_trackextr = df_trackextr[valid].reset_index(drop=True)

        # Fix local fIndexCollisions → global index. 
        df_trackextr["fIndexCollisions"] += collision_offset   
    
        # Save results in the container list.
        list_of_df.append( df_trackextr )
        
        # Update offset for next loop.
        collision_offset += len(df_coll)     
        
        
    # Create a unique pandas DataFrame by concatenating those in the container list. Events from dirrerent TTrees have non-overlapping
    # fIndexCollision thanks to the introduced collision_offset.
    df = pd.concat(list_of_df, ignore_index=True)

    # Keep only those particles whose charge is 1 or -1.
    df = df[df["fCharge"].isin([-1,1])]
    
    # Momenta columns: converting from track reference system to global reference system.
    df["px"] = df["fPt"] * np.cos(df["fAlpha"])
    df["py"] = df["fPt"] * np.sin(df["fAlpha"])
    df["pz"] = df["fPt"] * np.sinh(df["fEta"])
    
    # Energy columns: Ene for the D0 instance, and Anti-Ene for the Anti-D0 one.
    mass = np.where(df["fCharge"] > 0, m_pi, m_K)
    anti_mass = np.where(df["fCharge"] > 0, m_K, m_pi)
    df["Ene"] = np.sqrt((df["fPt"] * np.cosh(df["fEta"]))**2 + mass**2)
    df["Anti-Ene"] = np.sqrt((df["fPt"] * np.cosh(df["fEta"]))**2 + anti_mass**2)
    
    # Debug: checking that each process is running.
    pid =  os.getpid()
    print(f" From process {pid}: The starting dataframe has", len(df), "rows and ", len(df.columns), "columns.")
    memory = df.memory_usage(deep=True).sum() / (1024 ** 2)
    print(f"The starting dataframe occupies {memory:.2f} MB")
    # df.head()

    # List containers for variables found looping over relevant particle pairs.
    collision_indices = []
    dcaXY_products = []
    inv_masses = []
    anti_masses = []
    pt_totals = []
    pz_totals = []
    decay_lengths = []
    cos_pointings = []
    mother = []
    positive_tof_ka = []
    positive_tof_pi = []
    negative_tof_ka = []
    negative_tof_pi = []
    positive_tpc_ka = []
    positive_tpc_pi = []
    negative_tpc_ka = []
    negative_tpc_pi = []
    positive_pt = []
    negative_pt = []
    
    # Splitting the dataframe in positive and negative particles in order to make finding relevant pairs more easily.
    df_pos = df[ df['fCharge']>0 ]
    df_neg = df[ df['fCharge']<0 ]
    
    # Iterate over each collision group by finding all particles belonging to the same collision.
    for collision_idx in (df_neg['fIndexCollisions'].unique()):
        group_pos = df_pos[ df_pos['fIndexCollisions'] == collision_idx ]
        group_neg = df_neg[ df_neg['fIndexCollisions'] == collision_idx ]
    
        # Only collisions with at least a pair.
        if len(group_pos) < 1:   continue
    
        # Reset index of the group to 0..N-1 and move original index in new column 'orig_index'.
        group_pos = group_pos.reset_index().rename(columns={'index': 'orig_index'})
        group_neg = group_neg.reset_index().rename(columns={'index': 'orig_index'})
    
        # Indexes for all possible pairs.
        combinat = itertools.product( range(len(group_neg)), range(len(group_pos)) )
    
        # Iterate over all unique pairs of tracks.
        for combo in combinat:
            row_neg = group_neg.iloc[combo[0]]
            row_pos = group_pos.iloc[combo[1]]
    
            product_dcaXY = row_neg['fDcaXY'] * row_pos['fDcaXY']
            
            # Extracting variables for easier comprehension.
            pt1, pt2 = row_neg['fPt'], row_pos['fPt']
            E1 = row_neg['Ene']
            E2 = row_pos['Ene']
            anti_E1 = row_neg['Anti-Ene']
            anti_E2 = row_pos['Anti-Ene']
            px1 = row_neg["px"]
            py1 = row_neg["py"]
            pz1 = row_neg["pz"]
            px2 = row_pos["px"]
            py2 = row_pos["py"]
            pz2 = row_pos["pz"]

            # Invariant Mass computation in both D0 and anti-D0 cases.
            inv_mass = np.sqrt( (E1+E2)**2 - (px1+px2)**2 - (py1+py2)**2 - (pz1+pz2)**2 )
            anti_inv_mass = np.sqrt( (anti_E1+anti_E2)**2 - (px1+px2)**2 - (py1+py2)**2 - (pz1+pz2)**2 )

            # Keeping only those entries for which at least one between inv_mass and anti_inv_mass is in the desired mass range.
            # This is mainly to reduce the size of the final dataframe.
            is_within_mass_window = ( ((inv_mass > LOWER_MASS) & (inv_mass < UPPER_MASS)) | \
                                      ((anti_inv_mass > LOWER_MASS) & (anti_inv_mass < UPPER_MASS)) )
            if is_within_mass_window == False: continue

            # Assign mass of the mother based on the hypothesis on the daughters:
            # inv_mass has been calculated under the assumption that the positive particle is the Pion, so the mother is D0.
            mother_hyp = ""
            if (  ( (row_neg["Hyp"] in ["Kaon", "Both"]) & (row_pos["Hyp"] == "Pion") )  | \
                  ( (row_neg["Hyp"] == "Kaon" ) & (row_pos["Hyp"] in ["Pion", "Both"] ) )  ):
                mother_hyp = "D0"

            elif( ( (row_neg["Hyp"] == "Pion" ) & (row_pos["Hyp"] in ["Kaon", "Both"] ) )| \
                  ( (row_neg["Hyp"] in ["Pion", "Both"] ) & (row_pos["Hyp"]=="Kaon" ) ) ):
                mother_hyp = "Anti-D0"

            else:
                mother_hyp = "Undecided"
    
    
            # Total transverse momentum of the D0 candidate (used later for sliced plots).
            pt_total = np.sqrt((px1 + px2)**2 + (py1 + py2)**2)
    
            # Secondary vertex.
            SV_coords = np.array( secondary_vertex(row_neg, row_pos) )
    
            # Decay length: distance between PV and SV.
            PV_coords = np.array( [row_pos["fPosX"], row_pos["fPosY"], row_pos["fPosZ"]] )
            decay_lengths.append( np.linalg.norm(SV_coords - PV_coords ) )
    
            # Cosine of pointing angle (the angle between the direction of the mother particle and the line connecting PV and SV).
            mother_direction = [px1+px2, py1+py2, pz1+pz2]
            flight_line = SV_coords - PV_coords
            cos_pointings.append ( np.dot(mother_direction, flight_line)/(np.linalg.norm(mother_direction)*np.linalg.norm(flight_line)) )

            # Save single track variables to perform cuts in the future.
            p_pt = row_pos['fPt']
            n_pt = row_neg['fPt']
            p_tof_ka = row_pos['fNsigmaTOFka']
            p_tof_pi = row_pos['fNsigmaTOFpi']
            n_tof_ka = row_neg['fNsigmaTOFka']
            n_tof_pi = row_neg['fNsigmaTOFpi']
            p_tpc_ka = row_pos['fNsigmaTPCka']
            p_tpc_pi = row_pos['fNsigmaTPCpi']
            n_tpc_ka = row_neg['fNsigmaTPCka']
            n_tpc_pi = row_neg['fNsigmaTPCpi']

            # Add the found pairs to the lists.
            collision_indices.append(int(row_neg['fIndexCollisions']))
            dcaXY_products.append(product_dcaXY)
            inv_masses.append(inv_mass)
            anti_masses.append(anti_inv_mass)
            pt_totals.append(pt_total)
            pz_totals.append(pz1+pz2)
            mother.append(mother_hyp)
            positive_tof_ka.append(p_tof_ka)
            positive_tof_pi.append(p_tof_pi)
            negative_tof_ka.append(n_tof_ka)
            negative_tof_pi.append(n_tof_pi)
            positive_tpc_ka.append(p_tpc_ka)
            positive_tpc_pi.append(p_tpc_pi)
            negative_tpc_ka.append(n_tpc_ka)
            negative_tpc_pi.append(p_tpc_pi)
            positive_pt.append(p_pt)
            negative_pt.append(n_pt)
    
    
    # Create a dataframe with the final results to save to file for further analysis (forcing the data type to save space).
    df_pairs = pd.DataFrame({
        'collision_index': np.array(collision_indices, dtype="uint32"),
        'dcaXY_product': np.array(dcaXY_products, dtype="float32"),
        'inv_mass': np.array(inv_masses, dtype="float32"),
        'anti_mass': np.array(anti_masses, dtype="float32"),
        'pt': np.array(pt_totals, dtype="float32"),
        'pz': np.array(pz_totals, dtype="float32"),
        'decay_length': np.array(decay_lengths, dtype="float32"),
        'cos_pointing': np.array(cos_pointings, dtype="float32"),
        'particle': mother,
        'pos_tof_ka': np.array(positive_tof_ka, dtype="float32"),
        'pos_tof_pi': np.array(positive_tof_pi, dtype="float32"),
        'neg_tof_ka': np.array(negative_tof_ka, dtype="float32"),
        'neg_tof_pi': np.array(negative_tof_pi, dtype="float32"),
        'pos_tpc_ka': np.array(positive_tpc_ka, dtype="float32"),
        'pos_tpc_pi': np.array(positive_tpc_pi, dtype="float32"),
        'neg_tpc_ka': np.array(negative_tpc_ka, dtype="float32"),
        'neg_tpc_pi': np.array(negative_tpc_pi, dtype="float32"),
        'pos_pt': np.array(positive_pt, dtype="float32"),
        'neg_pt': np.array(negative_pt, dtype="float32"),
    })

    # Debug: check that each process has created the DataFrame.
    print("The final dataframe has", len(df_pairs), "rows")
    memory = df_pairs.memory_usage(deep=True).sum() / (1024 ** 2)
    print(f"The dataframe occupy {memory:.2f} MB")
    display( pd.concat([df_pairs.head(2),df_pairs.tail(3)]) )

    # We save later: if memory is not sufficient, save here (but this will cause to have overlapping collision_index between dataframes 
    # created by different processes: it is not really an issue, though, since it can be fixed later if needed).
    
    # save_name = "pairs_" + "_".join([which_chunk, which_number, str(J)])
    # df_pairs.to_pickle("/home/tpira/CombinatorialData/" + save_name + ".pkl")

    # print(f"{save_name} created.")

    return (df_pairs)

In [4]:
if __name__ == '__main__':

    # Define the number of processes
    N_PROCESSES = 3
    available_cores = mp.cpu_count()
    if (N_PROCESSES > available_cores):
        print("Warning: using more processes than cores.")

    # Set the cuts for particle identification: in this case the cuts are very broad to allow for further study after the pair computation
    # is over.

    # Kaon hypothesis.
    Ka_hyp = " ( fNsigmaTPCka.abs() < 3.0 ) & "\
             " ( \
                 ( (fPt <= 6.0) & (fNsigmaTOFka.abs() <= 8) ) | \
                 ( (fPt <= 6.0) & (fNsigmaTOFka == -999))\
                )"
    
    # Pion hypothesis.
    Pi_hyp = " ( fNsigmaTPCpi.abs() < 4.0 ) & "\
             " ( \
                 ( (fPt <= 6.0) & (fNsigmaTOFpi.abs() <= 12) ) | \
                 ( (fPt <= 6.0) & (fNsigmaTOFpi == -999) )\
                )"
    
    cut_expression = f"({Ka_hyp} | {Pi_hyp})"
    
    # Create a pool of processes.
    pool = mp.Pool(N_PROCESSES)
     
    # Set the path which the files are read from.
    #base_path = "/mnt/SingleTrackTrees/Data/alice_data_2023_LHC23f_535087_apass4_1300_Thinner/Output/Run535087"
    #base_path = "/home/pira/Documenti/PoD/LCP/LCP_B/ALICE/Alice_Data/Run535087/Run535087"
    base_path = "~/Downloads/"
    which_chunk = "Chunk1030"
    which_number = "006_011"
    which_file = "AO2Dtree.root"
    path = base_path + "/".join([which_chunk, which_number, which_file])

    # Create a text file to record which cuts where used.
    out_name = f"Cuts_for_{which_chunk}_{which_number}.txt"
    with open(out_name, "w") as f:
        f.write(cut_expression)

    start = time.time()
    # Let each process do the loading; load once to get number of files.
    N_FILES = 0
    with uproot.open(path) as file:
        N_FILES = len(file.keys(filter_name=r"*O2collision_001"))

    # This can be equal to the number of processes
    N_SPLITS = 9
    
    # Number of subsets of the data to create: this is due to memory problems when doing the combinatorial.
    subsets = np.array_split(range(0,N_FILES),N_SPLITS)
    
    results = pool.starmap(compute_pairs, [(path, subsets[i]) for i in range(len(subsets))])
    
    # starmap is a blocking statement, so the following will be executed only once the above computation is completed.
    
    # Get the results: each dataframe had its own "global" collision index: we need to make a unique global index.
    shifts = [np.max(results[i]["collision_index"]) for i in range(N_SPLITS)]
    global_shift = np.cumsum(shifts)

    for i in range(1,N_SPLITS):
        results[i]["collision_index"] += global_shift[i-1]

    for i in range(N_SPLITS):
        save_name = "par_pairs_" + "_".join([which_chunk, which_number, str(i)])
        results[i].to_pickle("Big_Combinatorial/" + save_name + ".pkl")

    # Stop the timer
    end = time.time()

    print()
    print(f'Time taken = {(end - start)/3600.0:.2f} h')

 From process 62175: The starting dataframe has 2602863 rows and  22 columns.
 From process 62173: The starting dataframe has 2623483 rows and  22 columns.
 From process 62174: The starting dataframe has 2602384 rows and  22 columns.
The starting dataframe occupies 359.93 MB
The starting dataframe occupies 362.78 MB
The starting dataframe occupies 359.87 MB
The final dataframe has 1656132 rows
The dataframe occupy 200.67 MB


,collision_index,dcaXY_product,inv_mass,anti_mass,pt,pz,decay_length,cos_pointing,particle,pos_tof_ka,pos_tof_pi,neg_tof_ka,neg_tof_pi,pos_tpc_ka,pos_tpc_pi,neg_tpc_ka,neg_tpc_pi,pos_pt,neg_pt
0,0,0.000018,1.615147,1.632913,0.444972,-0.379016,0.015624,0.293551,D0,-20.000397,-0.050894,-999.000000,-999.000000,-4.022532,-1.767226,-1.577024,-1.767226,0.698419,0.825447
1,0,-0.000011,1.871363,2.026030,2.238958,-1.313689,0.005953,-0.962537,Undecided,-20.000397,-0.050894,-999.000000,-999.000000,-4.022532,-1.767226,3.446402,-1.767226,0.698419,2.136394
1656129,539195,-0.000004,1.908885,2.125784,1.683693,-1.835615,0.005071,0.982950,D0,-999.000000,-999.000000,-0.605152,3.677589,-7.091856,-1.112971,2.771425,-1.112971,0.443117,2.016214
1656130,539195,-0.000015,1.969530,1.811795,1.069009,0.812923,0.014197,-0.784827,Anti-D0,-999.000000,-999.000000,-999.000000,-999.000000,1.744316,0.175417,-6.920839,0.175417,1.527228,0.459212
1656131,539195,0.000004,1.876546,1.696332,1.307405,0.561910,0.006684,-0.586922,Anti-D0,-999.000000,-999.000000,-999.000000,-999.000000,1.744316,0.175417,-7.818779,0.175417,1.527228,0.376923


 From process 62174: The starting dataframe has 2633753 rows and  22 columns.
The starting dataframe occupies 364.20 MB
The final dataframe has 1669347 rows
The dataframe occupy 202.27 MB


,collision_index,dcaXY_product,inv_mass,anti_mass,pt,pz,decay_length,cos_pointing,particle,pos_tof_ka,pos_tof_pi,neg_tof_ka,neg_tof_pi,pos_tpc_ka,pos_tpc_pi,neg_tpc_ka,neg_tpc_pi,pos_pt,neg_pt
0,1,1.610052e-06,1.847061,1.873596,1.261470,0.775846,0.002405,-0.120430,Undecided,-999.000000,-999.000000,-999.000000,-999.000000,-1.338372,-0.737560,0.311814,-0.737560,0.877350,1.231486
1,3,-2.041185e-07,2.119505,2.265686,2.111003,-1.480358,0.000691,-0.503690,D0,-24.156317,1.693317,-2.149143,2.054674,-2.177386,0.982025,-0.028490,0.982025,0.717203,1.976229
1669344,543129,9.651782e-06,1.596205,1.360049,2.137712,-0.761272,0.004811,0.740962,Anti-D0,-999.000000,-999.000000,-999.000000,-999.000000,-0.030273,-1.724565,-6.824761,-1.724565,1.950114,0.497381
1669345,543129,-4.749468e-06,2.065933,1.908381,1.828093,-0.606324,0.004001,0.297979,Anti-D0,-999.000000,-999.000000,-999.000000,-999.000000,-0.030273,-1.724565,-3.973881,-1.724565,1.950114,0.568691
1669346,543129,-5.891153e-07,1.931202,1.817061,1.011532,-0.708643,0.003411,0.409383,Anti-D0,-999.000000,-999.000000,-999.000000,-999.000000,0.900888,-0.296401,-3.973881,-0.296401,1.317730,0.568691


 From process 62173: The starting dataframe has 2582295 rows and  22 columns.
The starting dataframe occupies 357.09 MB
The final dataframe has 1657224 rows
The dataframe occupy 200.80 MB


...

The final dataframe has 1633134 rows
The dataframe occupy 197.88 MB


,collision_index,dcaXY_product,inv_mass,anti_mass,pt,pz,decay_length,cos_pointing,particle,pos_tof_ka,pos_tof_pi,neg_tof_ka,neg_tof_pi,pos_tpc_ka,pos_tpc_pi,neg_tpc_ka,neg_tpc_pi,pos_pt,neg_pt
0,3,0.000210,1.279043,1.701740,2.655652,-1.469882,0.079566,-0.884003,D0,-999.0,-999.0,-3.488489,-0.153022,-10.480336,1.713393,1.136394,1.713393,0.325400,2.360458
1,3,-0.000016,1.763097,2.069116,2.124529,-1.872319,0.006649,-0.278821,D0,-999.0,-999.0,-3.488489,-0.153022,-9.061459,0.829145,1.136394,0.829145,0.320041,2.360458
1633131,529863,0.000003,1.468711,1.575097,1.903372,-0.714195,0.003010,0.077643,D0,-999.0,-999.0,-999.000000,-999.000000,-3.201209,-0.854366,1.709506,-0.854366,0.717612,1.282260
1633132,529865,-0.000003,1.637618,1.617935,0.490780,0.909171,0.002553,-0.253211,Undecided,-999.0,-999.0,-999.000000,-999.000000,-0.774850,0.035122,-1.647001,0.035122,0.864392,0.671182
1633133,529865,-0.000006,1.590766,1.548329,0.514447,0.722693,0.005836,0.503611,D0,-999.0,-999.0,-999.000000,-999.000000,-0.774850,0.035122,1.329482,0.035122,0.864392,0.620597


The final dataframe has 1616276 rows
The dataframe occupy 195.84 MB


,collision_index,dcaXY_product,inv_mass,anti_mass,pt,pz,decay_length,cos_pointing,particle,pos_tof_ka,pos_tof_pi,neg_tof_ka,neg_tof_pi,pos_tpc_ka,pos_tpc_pi,neg_tpc_ka,neg_tpc_pi,pos_pt,neg_pt
0,0,-8.732158e-06,1.495236,1.401734,0.700385,0.423162,0.004480,0.228706,Anti-D0,-999.0,-999.0,-39.004692,0.562865,0.984832,1.678988,-7.190659,1.678988,0.806501,0.456469
1,1,9.790213e-08,1.661646,1.886205,1.688357,0.741452,0.001045,0.228320,D0,-999.0,-999.0,-2.916894,2.789417,-8.603320,0.390471,0.282685,0.390471,0.386575,1.894278
1616273,525513,9.675416e-07,1.665646,1.470959,2.533266,-1.684293,0.003838,0.297989,Undecided,-999.0,-999.0,-999.000000,-999.000000,1.945032,-0.022749,-1.329522,-0.022749,1.949854,0.765329
1616274,525513,2.207945e-06,1.763529,1.735804,0.605034,0.393280,0.010570,-0.411301,Undecided,-999.0,-999.0,-999.000000,-999.000000,-1.313535,-0.730043,-1.329522,-0.730043,0.885408,0.765329
1616275,525514,-5.687608e-06,1.397755,1.514322,0.883063,0.093473,0.008477,0.971878,D0,-999.0,-999.0,-999.000000,-999.000000,-7.899403,0.703159,0.298218,0.703159,0.369898,1.069735



Time taken = 1.96 h
